# Moondream Waste Detection – Colab Training

Notebook này chuẩn hóa quy trình train YOLO trên dataset `dataset_moondream`.

## 🚀 HƯỚNG DẪN NHANH (Step-by-step)

### Bước 1: Bật GPU (QUAN TRỌNG!)
1. Vào menu: `Runtime` → `Change runtime type`
2. Chọn `Hardware accelerator: GPU` (T4 hoặc A100)
3. Lưu và **restart runtime**

### Bước 2: Chạy các cell theo thứ tự

**Sau khi restart runtime, chạy lần lượt:**

1. **Cell 1** → Cài PyTorch 2.5.1 + torchvision + fix dependencies
   - ⚠️ **Sau khi chạy xong, RESTART RUNTIME lần nữa!**
   
2. **Cell 2** → Verify dependencies đã cài đúng chưa
   - Phải thấy: `✅ torchvision.ops.nms: OK`
   
3. **Cell 3** → Cài Ultralytics

4. **Cell 4** → Mount Google Drive (cần nhập auth code)

5. **Cell 6** → Copy dataset từ Drive

6. **Cell 8** → Train model (~1-2 giờ với GPU T4)

---

## ⏱️ Thời gian training ước tính:
- **CPU:** ~14-20 giờ ❌ (không khuyến nghị)
- **GPU T4:** ~1-2 giờ ✅
- **GPU A100:** ~30-45 phút ✅

---

## 🔧 Nếu gặp lỗi:

**Lỗi thường gặp:**
- `ImportError: cannot import name '_Ink'`
- `numpy X.X.X which is incompatible`
- `pillow X.X.X which is incompatible`

**Cách fix:**
1. **Restart runtime:** `Runtime` → `Restart runtime`
2. **Chạy lại Cell 1** (cell này tự động fix tất cả dependencies)
3. **Restart runtime** lần nữa
4. **Chạy Cell 2** để verify


In [ ]:
# Cài PyTorch 2.5.1 + torchvision + fix dependencies
# Cell này sẽ tự động fix tất cả dependency conflicts

print("🔧 Bắt đầu cài đặt và fix dependencies...")
print("=" * 60)

# Bước 1: Uninstall các package có conflict
print("\n1️⃣  Uninstalling conflicting packages...")
print("   (numpy, pillow, torch, torchvision, torchaudio, fsspec)")
%pip uninstall -y numpy pillow torch torchvision torchaudio fsspec

# Bước 2: Cài numpy và pillow đúng version
print("\n2️⃣  Installing compatible NumPy and Pillow...")
print("   NumPy: <2.1,>=1.26.0 (tương thích với numba, tensorflow, opencv)")
print("   Pillow: <12.0,>=8.0 (tương thích với gradio, torchvision)")
%pip install -q --no-cache-dir "numpy<2.1,>=1.26.0" "pillow<12.0,>=8.0"

# Bước 3: Fix fsspec (tương thích với datasets)
print("\n3️⃣  Installing compatible fsspec...")
print("   fsspec: <=2025.3.0,>=2023.1.0 (tương thích với datasets, gcsfs)")
%pip install -q --no-cache-dir "fsspec<=2025.3.0,>=2023.1.0"

# Bước 4: Cài PyTorch + torchvision
print("\n4️⃣  Installing PyTorch 2.5.1 + torchvision...")
%pip install -q --no-cache-dir torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --extra-index-url https://download.pytorch.org/whl/cu121

# Bước 5: Hoàn tất cài đặt
print("\n5️⃣  Installation completed!")
print("\n" + "=" * 60)
print("✅ Đã cài đặt:")
print("   - NumPy: <2.1,>=1.26.0 (tương thích với numba, tensorflow, opencv)")
print("   - Pillow: <12.0,>=8.0 (tương thích với gradio, torchvision)")
print("   - fsspec: <=2025.3.0,>=2023.1.0 (tương thích với datasets)")
print("   - PyTorch 2.5.1 + torchvision 0.20.1")
print("\n" + "=" * 60)
print("⚠️  QUAN TRỌNG: Phải restart runtime trước khi import!")
print("\n📋 BƯỚC TIẾP THEO:")
print("1. Restart runtime: Runtime → Restart runtime")
print("2. Chạy Cell 2 để verify (sẽ import và kiểm tra)")
print("3. Nếu Cell 2 OK → Tiếp tục các cell còn lại")
print("\n💡 Lý do: PyTorch không thể reload, phải restart runtime để load lại modules")


In [ ]:
# Verify dependencies (chạy SAU khi restart runtime)
import os

# Tắt W&B nếu không muốn dùng (bỏ comment dòng dưới)
# os.environ["WANDB_DISABLED"] = "true"

# Import và verify
import torch
import torchvision
import numpy as np
import PIL

print("🔍 Verifying installations...")
print("=" * 60)

print(f'\n✅ NumPy: {np.__version__} (phải < 2.1 để tương thích với numba)')
print(f'✅ Pillow: {PIL.__version__} (phải < 12.0)')
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ Torchvision: {torchvision.__version__}')
print(f'✅ CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✅ CUDA version: {torch.version.cuda}')
else:
    print("⚠️  CẢNH BÁO: Đang dùng CPU - training sẽ rất chậm (~14-20 giờ)!")
    print("   → Vào Runtime → Change runtime type → Chọn GPU")

# Kiểm tra torchvision::nms operator
try:
    from torchvision.ops import nms
    print('✅ torchvision.ops.nms: OK')
    print('\n' + "=" * 60)
    print('🎉 Tất cả dependencies đã được cài đặt đúng!')
except Exception as e:
    print(f'\n❌ Lỗi torchvision.ops.nms: {e}')
    print('\n🔧 Giải pháp:')
    print('1. Chạy Cell 1 (cài đặt dependencies)')
    print('2. Restart runtime: Runtime → Restart runtime')
    print('3. Chạy lại Cell 2 để verify')


In [ ]:
# Cài Ultralytics và các thư viện cần thiết
%pip install -q ultralytics==8.1.0 roboflow tqdm
print('✅ Đã cài đặt Ultralytics')


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


## 1. Copy dataset lên Colab

- Upload thư mục `dataset_moondream` (hoặc file `.zip`) vào Google Drive
- Cập nhật biến `DATASET_SOURCE` bên dưới trỏ tới vị trí đó
- Cell copy sẽ giải nén/sao chép sang `/content/dataset_moondream`


In [ ]:
import shutil
import yaml
import re
from pathlib import Path

DATASET_SOURCE = Path('/content/drive/MyDrive/dataset_moondream')
DATASET_TARGET = Path('/content/dataset_moondream')

if DATASET_SOURCE.suffix == '.zip':
    import zipfile
    print(f'Giải nén {DATASET_SOURCE} -> {DATASET_TARGET}')
    with zipfile.ZipFile(DATASET_SOURCE, 'r') as zf:
        zf.extractall('/content')
else:
    if DATASET_TARGET.exists():
        shutil.rmtree(DATASET_TARGET)
    print(f'Copy {DATASET_SOURCE} -> {DATASET_TARGET}')
    shutil.copytree(DATASET_SOURCE, DATASET_TARGET)

print('Done! Files:', len(list(DATASET_TARGET.rglob("*"))))

# Fix data.yaml format (đảm bảo names là list, không phải dict)
data_yaml_path = DATASET_TARGET / 'data.yaml'
if data_yaml_path.exists():
    print('\n🔧 Kiểm tra và sửa data.yaml...')
    
    # Đọc file như text để tránh lỗi parse
    with open(data_yaml_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Kiểm tra xem names có phải là dict format không (có dòng "  0: battery" hoặc " 10: trash")
    if re.search(r'^\s+\d+:', content, re.MULTILINE):
        print('   Phát hiện names là dict format, đang chuyển sang list...')
        
        # Tạo data.yaml mới với format đúng
        class_names = [
            'battery', 'biological', 'brown-glass', 'cardboard',
            'clothes', 'green-glass', 'metal', 'paper',
            'plastic', 'shoes', 'trash', 'white-glass'
        ]
        
        # Đọc các giá trị cũ nếu có
        path_match = re.search(r'^path:\s*(.+)$', content, re.MULTILINE)
        train_match = re.search(r'^train:\s*(.+)$', content, re.MULTILINE)
        val_match = re.search(r'^val:\s*(.+)$', content, re.MULTILINE)
        nc_match = re.search(r'^nc:\s*(\d+)$', content, re.MULTILINE)
        
        # Dùng absolute path để tránh lỗi resolve
        path_val = str(DATASET_TARGET.resolve())
        train_val = train_match.group(1).strip() if train_match else 'images/train'
        val_val = val_match.group(1).strip() if val_match else 'images/val'
        nc_val = nc_match.group(1).strip() if nc_match else '12'
        
        # Tạo YAML mới với list format và absolute path
        new_content = f"""path: {path_val}
train: {train_val}
val: {val_val}
nc: {nc_val}
names:
"""
        for name in class_names:
            new_content += f"  - {name}\n"
        
        # Ghi lại file
        with open(data_yaml_path, 'w', encoding='utf-8') as f:
            f.write(new_content)
        print('   ✅ Đã sửa data.yaml (chuyển names từ dict sang list)')
    else:
        # Thử parse để verify và fix path nếu cần
        try:
            with open(data_yaml_path, 'r', encoding='utf-8') as f:
                data = yaml.safe_load(f)
            
            # Kiểm tra và fix path nếu là relative
            if data.get('path') in ('.', './', ''):
                print('   Đang sửa path thành absolute path...')
                data['path'] = str(DATASET_TARGET.resolve())
                with open(data_yaml_path, 'w', encoding='utf-8') as f:
                    yaml.dump(data, f, default_flow_style=False, sort_keys=False)
                print('   ✅ Đã sửa path thành absolute path')
            
            if isinstance(data.get('names'), list):
                print('   ✅ data.yaml đã đúng format (names là list, path là absolute)')
            else:
                print('   ⚠️  data.yaml có format khác, nhưng có thể vẫn hoạt động')
        except Exception as e:
            print(f'   ⚠️  Không thể parse data.yaml: {e}')
            print('   → Sẽ tạo lại data.yaml với format đúng...')
            # Tạo lại file với format đúng
            class_names = [
                'battery', 'biological', 'brown-glass', 'cardboard',
                'clothes', 'green-glass', 'metal', 'paper',
                'plastic', 'shoes', 'trash', 'white-glass'
            ]
            new_content = f"""path: {str(DATASET_TARGET.resolve())}
train: images/train
val: images/val
nc: 12
names:
"""
            for name in class_names:
                new_content += f"  - {name}\n"
            with open(data_yaml_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print('   ✅ Đã tạo lại data.yaml với format đúng')
else:
    print('   ⚠️  Không tìm thấy data.yaml')


## 2. Train YOLO

Cập nhật các siêu tham số (epochs, batch, imgsz, model backbone) tùy nhu cầu. Kết quả sẽ nằm trong `runs/train/<name>`.


from ultralytics import YOLO
import torch

# Kiểm tra GPU và thiết lập device
if torch.cuda.is_available():
    device = '0'  # Sử dụng GPU đầu tiên (Tesla T4)
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU được phát hiện: {gpu_name}")
    print(f"   CUDA version: {torch.version.cuda}")
    print(f"   Device sẽ sử dụng: cuda:{device}")
else:
    device = 'cpu'
    print("⚠️  CẢNH BÁO: Không có GPU! Training sẽ rất chậm (~14-20 giờ)")
    print("   → Dừng cell này và bật GPU trước!")
    print("   Runtime → Change runtime type → GPU")

DATA_YAML = DATASET_TARGET / 'data.yaml'
MODEL_NAME = 'yolov8n.pt'  # hoặc đường dẫn checkpoint khác
EPOCHS = 50
IMGSZ = 640
BATCH = 16
RUN_NAME = 'moondream-colab'

# Tự động điều chỉnh batch size nếu dùng CPU
if device == 'cpu':
    BATCH = 8  # Giảm batch size cho CPU
    print(f"⚠️  Giảm batch size xuống {BATCH} cho CPU")

print(f'\n📊 Cấu hình training:')
print(f'   Dataset: {DATA_YAML}')
print(f'   Model: {MODEL_NAME}')
print(f'   Epochs: {EPOCHS}')
print(f'   Batch size: {BATCH}')
print(f'   Image size: {IMGSZ}')
print(f'   Device: {device} ({"GPU" if device != "cpu" else "CPU"})')
print(f'   Thời gian ước tính: {"~1-2 giờ" if device != "cpu" else "~14-20 giờ"}')

# Khởi tạo model
model = YOLO(MODEL_NAME)

# Train với device được chỉ định rõ ràng
print(f'\n🚀 Bắt đầu training trên {device.upper()}...')
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,  # ⚠️ QUAN TRỌNG: Ép sử dụng GPU
    name=RUN_NAME,
    project='runs/train'
)


## 3. Xem kết quả training

Sau khi train xong, kiểm tra metrics và plots trong thư mục `runs/train/<name>/`.


## 4. Validation và đánh giá model

Chạy validation trên tập test để xem độ chính xác của model.


In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_WEIGHTS = Path('runs/train') / RUN_NAME / 'weights' / 'best.pt'
LAST_WEIGHTS = Path('runs/train') / RUN_NAME / 'weights' / 'last.pt'

print(f'🔍 Đang validate model...')
print(f'   Best weights: {BEST_WEIGHTS}')

if BEST_WEIGHTS.exists():
    # Load model tốt nhất
    model = YOLO(str(BEST_WEIGHTS))
    
    # Chạy validation
    print('\n📊 Đang chạy validation...')
    metrics = model.val(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        device=device if 'device' in globals() else '0',
        plots=True  # Tạo confusion matrix, PR curve, etc.
    )
    
    # Hiển thị kết quả
    print('\n✅ Kết quả validation:')
    print(f'   mAP50: {metrics.box.map50:.4f}')
    print(f'   mAP50-95: {metrics.box.map:.4f}')
    print(f'   Precision: {metrics.box.mp:.4f}')
    print(f'   Recall: {metrics.box.mr:.4f}')
    
    print('\n📈 Chi tiết metrics:')
    print(json.dumps(metrics.results_dict, indent=2))
    
else:
    print('❗ Chưa tìm thấy best.pt – hãy kiểm tra tên run hoặc đợi train xong.')
    if LAST_WEIGHTS.exists():
        print(f'   → Tìm thấy last.pt, có thể dùng tạm: {LAST_WEIGHTS}')


## 5. Export model (tùy chọn)

Export model sang ONNX, TFLite, hoặc các format khác nếu cần deploy trên mobile/edge devices.


In [ ]:
from ultralytics import YOLO

BEST_WEIGHTS = Path('runs/train') / RUN_NAME / 'weights' / 'best.pt'

if BEST_WEIGHTS.exists():
    model = YOLO(str(BEST_WEIGHTS))
    
    # Export sang ONNX (khuyến nghị cho production)
    print('📦 Đang export sang ONNX...')
    onnx_path = model.export(
        format='onnx',
        imgsz=IMGSZ,
        opset=12,
        simplify=True  # Tối ưu model
    )
    print(f'✅ Đã export ONNX: {onnx_path}')
    
    # Export sang TFLite (cho mobile/Android nếu cần)
    # print('📦 Đang export sang TFLite...')
    # tflite_path = model.export(format='tflite', imgsz=IMGSZ)
    # print(f'✅ Đã export TFLite: {tflite_path}')
    
    # Export sang TensorRT (cho NVIDIA GPU nếu cần)
    # print('📦 Đang export sang TensorRT...')
    # trt_path = model.export(format='engine', imgsz=IMGSZ)
    # print(f'✅ Đã export TensorRT: {trt_path}')
    
else:
    print('❗ Chưa tìm thấy best.pt để export.')


## 6. Test model với ảnh mẫu (tùy chọn)

Test model với ảnh từ validation set để xem kết quả trực quan.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import random
from IPython.display import Image, display

BEST_WEIGHTS = Path('runs/train') / RUN_NAME / 'weights' / 'best.pt'

if BEST_WEIGHTS.exists():
    model = YOLO(str(BEST_WEIGHTS))
    
    # Tìm ảnh trong validation set
    val_images_dir = DATASET_TARGET / 'images' / 'val'
    if val_images_dir.exists():
        val_images = list(val_images_dir.glob('*.jpg')) + list(val_images_dir.glob('*.png'))
        
        if val_images:
            # Chọn ngẫu nhiên 3 ảnh
            test_images = random.sample(val_images, min(3, len(val_images)))
            
            print(f'🖼️  Đang test với {len(test_images)} ảnh từ validation set...\n')
            
            for img_path in test_images:
                print(f'📸 {img_path.name}')
                results = model.predict(
                    source=str(img_path),
                    imgsz=IMGSZ,
                    conf=0.25,  # Confidence threshold
                    save=True,
                    project='runs/predict',
                    name=f'{RUN_NAME}-test'
                )
                
                # Hiển thị ảnh kết quả
                result_img = Path('runs/predict') / f'{RUN_NAME}-test' / img_path.name
                if result_img.exists():
                    display(Image(str(result_img)))
                    print(f'   ✅ Đã lưu: {result_img}\n')
        else:
            print('⚠️  Không tìm thấy ảnh trong validation set')
    else:
        print('⚠️  Không tìm thấy thư mục validation images')
else:
    print('❗ Chưa tìm thấy best.pt để test.')


## 7. Lưu model về Google Drive

Copy model và các file cần thiết về Google Drive để tải về máy local và tích hợp vào backend.


In [ ]:
import shutil
from pathlib import Path

# Đường dẫn model đã train
BEST_WEIGHTS = Path('runs/train') / RUN_NAME / 'weights' / 'best.pt'
LAST_WEIGHTS = Path('runs/train') / RUN_NAME / 'weights' / 'last.pt'

# Đường dẫn ONNX (nếu đã export)
ONNX_PATH = BEST_WEIGHTS.parent.parent.parent / 'train' / RUN_NAME / 'weights' / f'{BEST_WEIGHTS.stem}.onnx'

# Thư mục đích trên Google Drive
DEST_DIR = Path('/content/drive/MyDrive/SmartSort/models')
DEST_DIR.mkdir(parents=True, exist_ok=True)

print(f'📦 Đang copy model về Google Drive...')
print(f'   Đích: {DEST_DIR}\n')

files_copied = []

# Copy best.pt (model tốt nhất)
if BEST_WEIGHTS.exists():
    target_best = DEST_DIR / 'best.pt'
    shutil.copy2(BEST_WEIGHTS, target_best)
    files_copied.append(('best.pt', target_best))
    print(f'✅ Đã copy: best.pt -> {target_best}')
    print(f'   Kích thước: {BEST_WEIGHTS.stat().st_size / (1024*1024):.2f} MB')
else:
    print('⚠️  Không tìm thấy best.pt')

# Copy last.pt (checkpoint cuối cùng)
if LAST_WEIGHTS.exists():
    target_last = DEST_DIR / 'last.pt'
    shutil.copy2(LAST_WEIGHTS, target_last)
    files_copied.append(('last.pt', target_last))
    print(f'✅ Đã copy: last.pt -> {target_last}')
    print(f'   Kích thước: {LAST_WEIGHTS.stat().st_size / (1024*1024):.2f} MB')

# Copy ONNX nếu có
if ONNX_PATH.exists():
    target_onnx = DEST_DIR / 'best.onnx'
    shutil.copy2(ONNX_PATH, target_onnx)
    files_copied.append(('best.onnx', target_onnx))
    print(f'✅ Đã copy: best.onnx -> {target_onnx}')
    print(f'   Kích thước: {ONNX_PATH.stat().st_size / (1024*1024):.2f} MB')

# Copy args.yaml và results.csv (metadata)
args_file = Path('runs/train') / RUN_NAME / 'args.yaml'
results_file = Path('runs/train') / RUN_NAME / 'results.csv'

if args_file.exists():
    shutil.copy2(args_file, DEST_DIR / 'args.yaml')
    print(f'✅ Đã copy: args.yaml')

if results_file.exists():
    shutil.copy2(results_file, DEST_DIR / 'results.csv')
    print(f'✅ Đã copy: results.csv')

print(f'\n📋 Tổng kết:')
print(f'   Đã copy {len(files_copied)} file(s) về Google Drive')
print(f'   Vị trí: {DEST_DIR}')
print(f'\n📥 BƯỚC TIẾP THEO:')
print(f'   1. Mở Google Drive trên máy tính')
print(f'   2. Vào thư mục: SmartSort/models/')
print(f'   3. Tải file best.pt về máy')
print(f'   4. Đặt vào: backend/models/best.pt')
print(f'   5. Restart backend để load model mới')


## 8. Hướng dẫn tích hợp model vào backend

Sau khi tải model về máy local, làm theo các bước sau:


### Bước 1: Tải model từ Google Drive

1. Mở Google Drive trên trình duyệt: https://drive.google.com
2. Vào thư mục: `SmartSort/models/`
3. Tải file `best.pt` về máy

### Bước 2: Đặt model vào backend

1. Tạo thư mục `models` trong backend (nếu chưa có):
   ```bash
   cd E:\Code\rac\backend
   mkdir models
   ```

2. Copy file `best.pt` vào `backend/models/best.pt`:
   ```bash
   # Windows PowerShell
   copy "C:\Users\YourName\Downloads\best.pt" "E:\Code\rac\backend\models\best.pt"
   ```

### Bước 3: Kiểm tra backend

Backend đã được cấu hình để tự động load `models/best.pt`. Kiểm tra:

1. Xem file `backend/api.py` dòng 30:
   ```python
   model_paths = [
       'models/best.pt',  # ← Model này sẽ được load
       ...
   ]
   ```

2. Restart backend:
   ```bash
   cd E:\Code\rac\backend
   python api.py
   # hoặc
   uvicorn api:app --host 0.0.0.0 --port 8000
   ```

3. Kiểm tra log khi start:
   ```
   ✅ Model loaded successfully: models/best.pt
   Model classes: ['battery', 'biological', 'brown-glass', ...]
   Total classes: 12
   SUCCESS: This is a waste classification model!
   ```

### Bước 4: Test API

```bash
curl http://localhost:8000/health
```

Response phải có:
- `model_loaded: true`
- `class_names` chứa 12 class Moondream
- `is_waste_model: true`

### ✅ Hoàn tất!

Model đã được tích hợp vào backend. App mobile sẽ tự động sử dụng model mới khi gọi API.
